# Name That LLM — UltraBERT v2 checkpoint 4 submission

Offline inference directly from the private Kaggle Model variation `zodrak/ultrabert380-v2/pytorch/checkpoint4-pytorch/1`. This is the current v2 training run’s selected best checkpoint at epoch 4 (validation accuracy 0.555556).

In [ ]:
from pathlib import Path
import json, os

MODEL_INPUT = Path('/kaggle/input/models/zodrak/ultrabert380-v2/pytorch/checkpoint4-pytorch/1')
MODEL_DIR = MODEL_INPUT
TEST_CANDIDATES = [Path('/kaggle/input/competitions/name-that-llm/test.csv'), Path('/kaggle/input/name-that-llm/test.csv')]
OUTPUT = Path('/kaggle/working/submission.parquet')
EXPECTED_EXPERIMENT_ID = 'ultrabert380_v2_data88cea686_split59c9631c_seed2026_cfg5635d548_code667e2d0'
EXPECTED_EPOCH = 4
EXPECTED_VALIDATION_ACCURACY = 0.5555555555555556

assert MODEL_DIR.is_dir(), f'Attached Kaggle model files not found: {MODEL_DIR}'
assert (MODEL_DIR / 'model.safetensors').is_file(), 'UltraBERT weights are missing'
assert (MODEL_DIR / 'tokenizer.json').is_file(), 'UltraBERT tokenizer is missing'
TEST_CSV = next((path for path in TEST_CANDIDATES if path.is_file()), None)
assert TEST_CSV is not None, f'Competition test.csv not found in: {TEST_CANDIDATES}'
model_config = json.loads((MODEL_DIR / 'config.json').read_text())
classes = [model_config['id2label'][str(i)] for i in range(len(model_config['id2label']))]
selection = json.loads((MODEL_DIR / 'selection.json').read_text())
assert selection['epoch'] == EXPECTED_EPOCH, selection
assert abs(selection['value'] - EXPECTED_VALIDATION_ACCURACY) < 1e-12, selection

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
import torch
print({'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'cuda_available': torch.cuda.is_available()})
assert torch.cuda.is_available(), 'Kaggle did not provide a CUDA GPU; aborting before inference.'
print({'experiment_id': EXPECTED_EXPERIMENT_ID, 'selected_epoch': selection['epoch'],
       'validation_accuracy': selection['value'], 'model_path': str(MODEL_DIR), 'device': 'cuda', 'test': str(TEST_CSV)})

In [ ]:
import numpy as np
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

test = pd.read_csv(TEST_CSV, dtype={'id': 'string'})
assert {'id', 'text'}.issubset(test.columns), test.columns.tolist()
assert test['id'].notna().all() and not test['id'].duplicated().any()
assert test['text'].notna().all() and test['text'].astype(str).str.strip().ne('').all()

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR, local_files_only=True, trust_remote_code=True
).to('cuda').eval()
predicted = []
with torch.inference_mode():
    for start in range(0, len(test), 8):
        batch = test['text'].iloc[start:start + 8].astype(str).tolist()
        tokens = tokenizer(batch, truncation=True, max_length=384, padding=True, return_tensors='pt')
        tokens = {name: value.to('cuda') for name, value in tokens.items()}
        logits = model(**tokens).logits
        predicted.extend(logits.argmax(dim=-1).cpu().tolist())

submission = pd.DataFrame({'id': test['id'], 'model': np.asarray(classes)[predicted]})
assert submission.columns.tolist() == ['id', 'model']
assert len(submission) == len(test)
assert submission['id'].astype(str).tolist() == test['id'].astype(str).tolist()
assert submission['model'].notna().all() and submission['model'].isin(classes).all()
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
submission.to_parquet(OUTPUT, index=False)
print({'rows': len(submission), 'output': str(OUTPUT), 'model': 'ultrabert380',
       'experiment_id': EXPECTED_EXPERIMENT_ID, 'selected_epoch': EXPECTED_EPOCH,
       'selected_validation_accuracy': EXPECTED_VALIDATION_ACCURACY})
display(submission.head())